# judgeaudit — 60-second demo

Renders a full **judge reliability report card** from the small grade samples bundled in this
repo (`demo_grades.jsonl`, `demo_grades_track_b.jsonl`). No API key, no data download required. 
It reads committed grades and computes everything locally in well under a minute.

> These numbers come from a *small* bundled sample (45 conversations for agreement, 10 examples
> for the variance grid), so they differ from the full-run report card in the README. The point
> here is that the tool runs end-to-end on a fresh clone.


In [1]:
import pandas as pd
from judgeaudit import report_card

track_a = pd.read_json("demo_grades.jsonl", lines=True)          # judge vs physician
track_b = pd.read_json("demo_grades_track_b.jsonl", lines=True)  # model-comparison grid
md = report_card(track_a, track_b)

try:                                    
    from IPython.display import Markdown, display
    display(Markdown(md))
except ImportError:
    print(md)

# Judge Reliability Report Card

## 1. Agreement with human labels

| judge | pct_agree | kappa | macro_f1 |
|---|---|---|---|
| openai/gpt-4.1 | 0.881 | 0.623 | 0.809 |
| anthropic/claude-sonnet-4.5 | 0.836 | 0.607 | 0.801 |
| google/gemini-2.5-flash | 0.806 | 0.302 | 0.633 |

## 2. Uncertainty (clustered 95% CIs)

- anthropic/claude-sonnet-4.5: agreement 0.836 [0.730, 0.925]

- google/gemini-2.5-flash: agreement 0.806 [0.696, 0.896]

- openai/gpt-4.1: agreement 0.881 [0.779, 0.958]

## 3. Sensitivity (movement in score from config choices)

| source | spread | vs model gap |
|---|---|---|
| sampling reps | 0.053 | 21% |
| judge choice | 0.139 | 55% |
| prompt wording | 0.035 | 14% |
| **MODEL GAP** | **0.252** | — |

## 4. Power

- n=10: minimum detectable gap ~0.299 at 80% power. Observed model gap 0.252 is **NOT detectable** at this n.

The same thing from the command line (after `pip install -e .`):

```
judgeaudit report demo_grades.jsonl                                                     # sections 1-2
judgeaudit report demo_grades.jsonl --track-b demo_grades_track_b.jsonl -o report.md    # all 4
```

**Use it on your own eval:** pass any grades file with the same row schema.
Track A rows need `judge_model, variant, grade, physician_label, prompt_id`
The optional Track B grid needs `response_model, judge_model, variant, run_tag, criterion_idx, points, grade, prompt_id`.
